# lm_eval로 모델 평가하기

이 노트북에서는 [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) 라이브러리를 사용해 양자화된 모델과 양자화되지 않은 모델을 평가하고 비교하는 방법을 보여 줍니다.

두 모델을 대상으로 MMLU(Massive Multitask Language Understanding) 벤치마크를 실행하고 성능을 비교합니다.

**참고:** 여러 사용자가 동시에 이 테스트를 실행할 수 있으므로, 모델 엔드포인트에 과부하가 걸리지 않도록 요청 속도를 제한합니다.

---
## 0. 설정

모델 엔드포인트를 설정합니다.

In [ ]:
import os

# 모델 엔드포인트 - 본인의 환경에 맞게 설정하세요
os.environ["UNQUANTIZED_URL"] = "http://llama-32-predictor.ai501.svc.cluster.local:8080"
os.environ["UNQUANTIZED_MODEL"] = "llama32"

os.environ["QUANTIZED_URL"] = "http://llama-32-fp8-predictor.ai501.svc.cluster.local:8080"
os.environ["QUANTIZED_MODEL"] = "llama32-fp8"

print(f"Unquantized endpoint: {os.environ['UNQUANTIZED_URL']}")
print(f"Unquantized model: {os.environ['UNQUANTIZED_MODEL']}")
print(f"Quantized endpoint: {os.environ['QUANTIZED_URL']}")
print(f"Quantized model: {os.environ['QUANTIZED_MODEL']}")

---
## 1. lm_eval 이해하기

`lm-evaluation-harness`는 언어 모델을 평가하는 표준 프레임워크입니다. 다음을 지원합니다.

- MMLU, HellaSwag, ARC, TruthfulQA 등을 포함한 **200개 이상의 벤치마크**
- 로컬 모델, OpenAI API, vLLM 엔드포인트 등 **다양한 모델 백엔드**
- 재현 가능한 비교를 위한 **표준화된 평가**

OpenAI 호환 API(vLLM 등)로 서빙되는 원격 모델에는 `local-completions` 모델 유형을 사용합니다.

---
## 2. 양자화되지 않은 모델 평가하기

먼저 양자화되지 않은(원래 정밀도의) 모델에 대해 MMLU 평가를 실행해 보겠습니다.

워크숍에서 빠르게 시연할 수 있도록 MMLU 태스크 하나와 제한된 샘플만 사용합니다. 프로덕션 평가에서는 `limit` 파라미터를 제거하고 태스크를 더 추가하세요.

In [ ]:
import lm_eval
import lm_eval.models.openai_completions
from lm_eval.models.openai_completions import LocalCompletionsAPI

# lm_eval의 openai_completions 모듈에서 tqdm import가 누락된 문제 수정
from tqdm import tqdm
lm_eval.models.openai_completions.tqdm = tqdm

# 모듈의 전역 네임스페이스에도 주입
import sys
sys.modules['lm_eval.models.openai_completions'].__dict__['tqdm'] = tqdm

# 양자화되지 않은 모델 설정
unquantized_model = LocalCompletionsAPI(
    model=os.environ["UNQUANTIZED_MODEL"],
    base_url=f"{os.environ['UNQUANTIZED_URL']}/v1/completions",
    num_concurrent=1,  # 과부하를 막기 위해 동시 요청 수 제한
    tokenizer_backend="huggingface",
    tokenizer="RedHatAI/Meta-Llama-3.1-8B-Instruct-quantized.w4a16",
)

print("Unquantized model configured.")

In [ ]:
# 양자화되지 않은 모델에 대해 MMLU 평가 실행
# 워크숍에서 빠르게 시연하기 위해 샘플 수를 제한
unquantized_results = lm_eval.simple_evaluate(
    model=unquantized_model,
    tasks=["mmlu_abstract_algebra"],  # 속도를 위해 태스크 하나만 사용
    num_fewshot=0,                     # few-shot 예시 없음(더 빠름)
    batch_size=1,
    limit=50,                          # 50개 샘플만 평가
)

print("Unquantized model evaluation complete.")

In [ ]:
# 양자화되지 않은 모델의 결과 표시
print("Unquantized Model Results")
print("=" * 50)
for task, metrics in unquantized_results["results"].items():
    acc = metrics.get("acc,none", metrics.get("acc", "N/A"))
    if isinstance(acc, float):
        print(f"{task}: {acc:.4f}")
    else:
        print(f"{task}: {acc}")

---
## 3. 양자화된 모델 평가하기

이제 FP8로 양자화된 모델에 대해 같은 평가를 실행해 보겠습니다.

In [ ]:
# 양자화된 모델 설정
quantized_model = LocalCompletionsAPI(
    model=os.environ["QUANTIZED_MODEL"],
    base_url=f"{os.environ['QUANTIZED_URL']}/v1/completions",
    num_concurrent=1,  # 과부하를 막기 위해 동시 요청 수 제한
    tokenizer_backend="huggingface",
    tokenizer="RedHatAI/Meta-Llama-3.1-8B-Instruct-quantized.w4a16",
)

print("Quantized model configured.")

In [ ]:
# 양자화된 모델에 대해 MMLU 평가 실행
# 비교를 위해 같은 수의 제한된 샘플 사용
quantized_results = lm_eval.simple_evaluate(
    model=quantized_model,
    tasks=["mmlu_abstract_algebra"],  # 속도를 위해 태스크 하나만 사용
    num_fewshot=0,                     # few-shot 예시 없음(더 빠름)
    batch_size=1,
    limit=50,                          # 50개 샘플만 평가
)

print("Quantized model evaluation complete.")

In [ ]:
# 양자화된 모델의 결과 표시
print("Quantized Model Results")
print("=" * 50)
for task, metrics in quantized_results["results"].items():
    acc = metrics.get("acc,none", metrics.get("acc", "N/A"))
    if isinstance(acc, float):
        print(f"{task}: {acc:.4f}")
    else:
        print(f"{task}: {acc}")

---
## 4. 결과 비교하기

두 모델의 성능을 나란히 비교해 보겠습니다.

In [ ]:
# 결과 비교
print("Model Comparison: Unquantized vs Quantized (FP8)")
print("=" * 70)
print(f"{'Task':<30} {'Unquantized':>15} {'Quantized':>15} {'Diff':>10}")
print("-" * 70)

total_unquant = 0
total_quant = 0
num_tasks = 0

for task in unquantized_results["results"].keys():
    unquant_acc = unquantized_results["results"][task].get("acc,none", 0)
    quant_acc = quantized_results["results"][task].get("acc,none", 0)
    
    if isinstance(unquant_acc, float) and isinstance(quant_acc, float):
        diff = quant_acc - unquant_acc
        print(f"{task:<30} {unquant_acc:>15.4f} {quant_acc:>15.4f} {diff:>+10.4f}")
        total_unquant += unquant_acc
        total_quant += quant_acc
        num_tasks += 1

if num_tasks > 0:
    avg_unquant = total_unquant / num_tasks
    avg_quant = total_quant / num_tasks
    avg_diff = avg_quant - avg_unquant
    print("-" * 70)
    print(f"{'Average':<30} {avg_unquant:>15.4f} {avg_quant:>15.4f} {avg_diff:>+10.4f}")
    print()
    print(f"Quantization impact: {avg_diff*100:+.2f}% accuracy change")

---
## 요약

배운 내용:

- `lm_eval`로 OpenAI 호환 API를 통해 서빙되는 모델을 평가하는 방법
- 원격 모델 엔드포인트에서 MMLU 벤치마크를 실행하는 방법
- 양자화된 모델과 양자화되지 않은 모델의 성능을 비교하는 방법
- 여러 사용자가 엔드포인트를 공유할 때 요청 속도를 제한하는 모범 사례

**핵심 정리:**
- FP8 양자화는 보통 정확도 손실이 매우 적습니다(대개 1% 미만)
- 그 대가로 메모리 사용량과 추론 속도가 크게 개선됩니다
- 항상 본인의 사용 사례와 관련된 태스크로 벤치마크하세요